# 🚀 LENTERA Fine-Tuning - Phase 2

Fine-tune Llama-3-8B with QLoRA using Axolotl framework

## 📋 Requirements:
- ✅ Dataset: train.jsonl & val.jsonl (from Phase 1)
- ✅ GPU: T4 (Colab Free) or A100 (Colab Pro)
- ✅ Time: 2-4 hours (T4) or 1-2 hours (A100)
- ✅ Stable internet connection

**IMPORTANT**: Keep this tab open during training!

---
## 🔧 Step 1: Check GPU

In [ ]:
!nvidia-smi
print("\n✅ GPU Status verified")

---
## 📤 Step 2: Upload Training Files

**Action Required**: Upload these files from `C:\LenteraDreamFlow\backend\finetuning\`:
1. `train.jsonl` (~750KB, 1,332 samples)
2. `val.jsonl` (~81KB, 148 samples)

Use the folder icon on left sidebar → Upload button

In [ ]:
# Verify files uploaded
import os
import json

files_ok = True

if os.path.exists('train.jsonl'):
    with open('train.jsonl') as f:
        train_count = sum(1 for _ in f)
    print(f"✅ train.jsonl found: {train_count} samples")
else:
    print("❌ train.jsonl not found! Please upload.")
    files_ok = False

if os.path.exists('val.jsonl'):
    with open('val.jsonl') as f:
        val_count = sum(1 for _ in f)
    print(f"✅ val.jsonl found: {val_count} samples")
else:
    print("❌ val.jsonl not found! Please upload.")
    files_ok = False

if files_ok:
    print(f"\n🎯 Total: {train_count + val_count} samples ready for training!")

---
## 🛠️ Step 3: Install Axolotl Framework

In [ ]:
%%capture
# Install dependencies (silent mode)
!pip install -U pip
!pip install torch torchvision torchaudio
!pip install transformers accelerate peft bitsandbytes
!pip install datasets trl

print("✅ All dependencies installed")

In [ ]:
# Clone Axolotl
!git clone https://github.com/OpenAccess-AI-Collective/axolotl
%cd axolotl
!pip install -e .
print("✅ Axolotl installed")

---
## ⚙️ Step 4: Configuration

Create training config based on your `lentera_config.yaml`

In [ ]:
%%writefile lentera_config.yaml
base_model: meta-llama/Meta-Llama-3-8B
model_type: LlamaForCausalLM
tokenizer_type: LlamaTokenizer

load_in_8bit: false
load_in_4bit: true
strict: false

datasets:
  - path: ../train.jsonl
    type: sharegpt
    conversation: chatml
  - path: ../val.jsonl
    type: sharegpt
    conversation: chatml

dataset_prepared_path:
val_set_size: 0.1
output_dir: ./lentera-finetuned

sequence_len: 2048
sample_packing: true
pad_to_sequence_len: true

adapter: qlora
lora_r: 32
lora_alpha: 16
lora_dropout: 0.05
lora_target_modules:
lora_target_linear: true
lora_fan_in_fan_out:

wandb_project:
wandb_entity:
wandb_watch:
wandb_name:
wandb_log_model:

gradient_accumulation_steps: 4
micro_batch_size: 2
num_epochs: 3
optimizer: adamw_bnb_8bit
lr_scheduler: cosine
learning_rate: 0.0002

train_on_inputs: false
group_by_length: false
bf16: auto
fp16:
tf32: false

gradient_checkpointing: true
early_stopping_patience:
resume_from_checkpoint:
local_rank:
logging_steps: 10
xformers_attention:
flash_attention: true

warmup_steps: 10
evals_per_epoch: 4
eval_table_size:
eval_max_new_tokens: 128
saves_per_epoch: 1
debug:
deepspeed:
weight_decay: 0.0
fsdp:
fsdp_config:
special_tokens:
  pad_token: "<|end_of_text|>"

---
## 🚀 Step 5: Start Training

**⏱️ Expected Duration**: 2-4 hours (T4 GPU)

**IMPORTANT**: 
- Keep this tab open!
- Laptop must stay on (can minimize browser)
- WiFi must stay connected

In [ ]:
# Prepare dataset
!accelerate launch -m axolotl.cli.preprocess lentera_config.yaml

In [ ]:
# Start training
!accelerate launch -m axolotl.cli.train lentera_config.yaml

print("\n🎉 Training complete!")

---
## 📊 Step 6: Check Results

In [ ]:
# List checkpoints
!ls -lh lentera-finetuned/

# Check final checkpoint
import os
checkpoints = [d for d in os.listdir('lentera-finetuned') if 'checkpoint' in d]
if checkpoints:
    print(f"\n✅ Found {len(checkpoints)} checkpoint(s)")
    print(f"Latest: {sorted(checkpoints)[-1]}")
else:
    print("❌ No checkpoints found")

---
## 💾 Step 7: Download Model

In [ ]:
# Zip the model for download
!zip -r lentera-model.zip lentera-finetuned/

from google.colab import files
print("📥 Downloading model (this may take a few minutes)...")
files.download('lentera-model.zip')
print("✅ Download started!")

---
## 🧪 Step 8: Quick Test

In [ ]:
# Test the model
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load model
model_path = "lentera-finetuned"
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Test prompt
prompt = "Aku stress banget kuliah, deadlines numpuk semua"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"User: {prompt}")
print(f"\nLENTERA: {response}")

---
## ✅ DONE!

**Next Steps**:
1. ✅ Download `lentera-model.zip`
2. ✅ Extract locally
3. ✅ Convert to GGUF format (Phase 3)
4. ✅ Deploy to Ollama

**Model Location**: `C:\LenteraDreamFlow\models\lentera-finetuned\`